# YOLOv12 Character-Level License Plate Training (Google Drive Safe)

## Overview
Train **YOLOv12** model (latest YOLO version) to detect:
- Individual characters (0-9, A-Z)
- Plate boundaries
- Plate types (new_DUBAI, old_DUBAI, etc.)

**Dataset:** 11,650 images with 51 classes

**Model:** YOLOv12n (nano - fastest, latest architecture)

**Expected Training Time:** 2-3 hours on Google Colab T4 GPU

**🔒 IMPORTANT FEATURES:**
- ✅ **All outputs saved to Google Drive** (survives Colab disconnects)
- ✅ **Auto-checkpointing** every few epochs
- ✅ **Resume training** if interrupted
- ✅ **Keep-alive** to prevent disconnects
- ✅ **Final models auto-saved** to Drive

## Step 1: Setup Environment & Mount Google Drive

**⚠️ CRITICAL: Run this cell first to mount Google Drive!**

In [ ]:
# Mount Google Drive FIRST (so all outputs are saved there)
from google.colab import drive
drive.mount('/content/drive')

# Create project directory in Google Drive
import os
project_dir = '/content/drive/MyDrive/YOLO_PlateScanner_Training'
os.makedirs(project_dir, exist_ok=True)
os.makedirs(f'{project_dir}/models', exist_ok=True)
os.makedirs(f'{project_dir}/results', exist_ok=True)

print("✅ Google Drive mounted")
print(f"✅ Project directory: {project_dir}")
print(f"\n📁 This is where ALL your models will be saved!")
print(f"   Even if Colab disconnects, your models are safe in Drive.")

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install latest Ultralytics (with YOLOv12 support)
!pip install ultralytics>=8.3.0 -q

# Verify installation and YOLOv12 availability
from ultralytics import YOLO
import torch

print(f"✅ Ultralytics installed")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# Check available YOLO models
print(f"\n📋 Checking YOLOv12 availability...")
try:
    test_model = YOLO('yolo12n.pt')
    print("✅ YOLOv12 is available!")
except Exception as e:
    print(f"ℹ️ YOLOv12 will be downloaded during training")

## Step 2: Keep-Alive Function (Prevents Disconnects)

**Run this to keep Colab from disconnecting during long training**

In [ ]:
# Keep Colab alive during long training
from IPython.display import display, Javascript

js_code = '''
function KeepClicking(){
    console.log("Keeping the session alive...");
    document.querySelector("colab-connect-button").click();
}
setInterval(KeepClicking, 60000); // Click every 60 seconds
'''

display(Javascript(js_code))
print("✅ Keep-alive activated!")
print("   Colab will stay connected during training.")

## Step 3: Upload & Extract Dataset

### Upload Your Dataset to Google Drive

**Instructions:**
1. On your Mac, compress your dataset:
   ```bash
   cd /Users/anishkumar/Desktop/Anish/Projects/Personal/Trignova/Projects/Source/PlateScanner/Model
   zip -r car_plate_dataset.zip "car plate images"
   ```

2. Upload `car_plate_dataset.zip` to your Google Drive

3. Update the path below

In [ ]:
# Extract dataset from Google Drive
import zipfile
import os

# UPDATE THIS PATH to where you uploaded the zip file in Google Drive
dataset_zip_path = "/content/drive/MyDrive/car_plate_dataset.zip"  # ← UPDATE THIS

# Check if file exists
if not os.path.exists(dataset_zip_path):
    print(f"❌ File not found: {dataset_zip_path}")
    print(f"\n📋 Please:")
    print(f"   1. Upload car_plate_dataset.zip to Google Drive")
    print(f"   2. Update the path above")
    print(f"   3. Re-run this cell")
else:
    # Extract dataset to /content (faster for training)
    print(f"✅ Found dataset: {dataset_zip_path}")
    print(f"📦 Extracting... (this may take a few minutes)")
    
    with zipfile.ZipFile(dataset_zip_path, 'r') as zip_ref:
        zip_ref.extractall("/content/dataset")
    
    print("✅ Dataset extracted to /content/dataset")
    !ls -lh /content/dataset

## Step 4: Prepare Dataset Configuration

In [ ]:
# Create corrected data.yaml with absolute paths
data_yaml_content = """# YOLOv12 Character-Level License Plate Dataset

# Paths (absolute)
path: /content/dataset/car plate images
train: train/images
val: valid/images
test: test/images

# Number of classes
nc: 51

# Class names
names: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'exp', 'new_DUBAI', 'new_RAK', 'new_abudabi', 'new_ajman', 'new_am', 'new_fujairah', 'old_DUBAI', 'old_RAK', 'old_abudabi', 'old_ajman', 'old_am', 'old_fujira', 'old_sharka', 'plate']
"""

# Write corrected data.yaml
with open('/content/dataset/car plate images/data_corrected.yaml', 'w') as f:
    f.write(data_yaml_content)

print("✅ data_corrected.yaml created")
!cat "/content/dataset/car plate images/data_corrected.yaml"

In [ ]:
# Verify dataset structure
import os

base_path = "/content/dataset/car plate images"

train_imgs = len(os.listdir(f"{base_path}/train/images"))
train_labels = len(os.listdir(f"{base_path}/train/labels"))

val_imgs = len(os.listdir(f"{base_path}/valid/images"))
val_labels = len(os.listdir(f"{base_path}/valid/labels"))

test_imgs = len(os.listdir(f"{base_path}/test/images"))
test_labels = len(os.listdir(f"{base_path}/test/labels"))

print("📊 Dataset Summary:")
print(f"   Train: {train_imgs} images, {train_labels} labels")
print(f"   Valid: {val_imgs} images, {val_labels} labels")
print(f"   Test:  {test_imgs} images, {test_labels} labels")
print(f"   Total: {train_imgs + val_imgs + test_imgs} images")

if train_imgs != train_labels or val_imgs != val_labels or test_imgs != test_labels:
    print("\n⚠️ Warning: Image/label count mismatch!")
else:
    print("\n✅ Dataset verified - ready for YOLOv12 training")

## Step 5: Train YOLOv12 Model (SAVES TO GOOGLE DRIVE)

**🔒 IMPORTANT:** 
- Training will save to `/content/drive/MyDrive/YOLO_PlateScanner_Training/`
- Even if Colab disconnects, your models are SAFE in Google Drive!
- Expected time: 2-3 hours

**⚠️ DO NOT CLOSE THIS TAB during training!**

In [ ]:
# Load pre-trained YOLOv12n model
model = YOLO('yolo12n.pt')  # Latest YOLOv12 nano model

print("✅ YOLOv12n model loaded")
print(f"   Starting training with 51 classes...")
print(f"   Using latest YOLO architecture with enhanced performance")
print(f"\n🔒 All outputs will be saved to Google Drive!")
print(f"   Project directory: {project_dir}")

In [ ]:
# Train YOLOv12 model with Google Drive output
import os
from datetime import datetime

# Set output directory to Google Drive
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_name = f'uae_character_detector_v12_{timestamp}'
project_save_path = f'{project_dir}/results'

print(f"📁 Training will save to: {project_save_path}/{output_name}")
print(f"🚀 Starting training...\n")

results = model.train(
    data='/content/dataset/car plate images/data_corrected.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    
    # SAVE TO GOOGLE DRIVE
    project=project_save_path,
    name=output_name,
    
    # Training settings
    patience=20,  # Early stopping
    save=True,    # Save checkpoints
    save_period=10,  # Save checkpoint every 10 epochs
    plots=True,
    device=0,  # GPU
    workers=8,
    
    # YOLOv12-optimized augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10,
    translate=0.1,
    scale=0.5,
    fliplr=0.0,
    mosaic=1.0,
    
    # YOLOv12 specific
    optimizer='auto',
    close_mosaic=10,
    
    # Resume training if interrupted
    resume=False,  # Set to True if resuming
    exist_ok=True
)

print("\n" + "="*60)
print("✅ YOLOv12 training complete!")
print("="*60)
print(f"Best model saved at: {project_save_path}/{output_name}/weights/best.pt")
print(f"\n🔒 Your model is safe in Google Drive!")
print(f"   You can access it anytime at: {project_save_path}/{output_name}")

## Step 6: Copy Best Model to Safe Location

**This ensures you have a backup of your best model**

In [ ]:
# Copy best model to a permanent location in Google Drive
import shutil
import os

# Find the most recent training run
training_runs = [d for d in os.listdir(project_save_path) if d.startswith('uae_character_detector')]
if training_runs:
    latest_run = max(training_runs, key=lambda x: os.path.getmtime(os.path.join(project_save_path, x)))
    best_model_path = f'{project_save_path}/{latest_run}/weights/best.pt'
    last_model_path = f'{project_save_path}/{latest_run}/weights/last.pt'
    
    # Copy to models directory
    if os.path.exists(best_model_path):
        shutil.copy(best_model_path, f'{project_dir}/models/character_detector_v12_best.pt')
        print(f"✅ Best model copied to: {project_dir}/models/character_detector_v12_best.pt")
    
    if os.path.exists(last_model_path):
        shutil.copy(last_model_path, f'{project_dir}/models/character_detector_v12_last.pt')
        print(f"✅ Last model copied to: {project_dir}/models/character_detector_v12_last.pt")
    
    # Save the path for later cells
    model_dir = f'{project_save_path}/{latest_run}'
    print(f"\n📁 Your training directory: {model_dir}")
    print(f"\n🎉 Models are now safely backed up in Google Drive!")
else:
    print("❌ No training runs found!")

## Step 7: Validate Model (Load from Google Drive)

In [ ]:
# Load best model from Google Drive for validation
best_model = YOLO(f'{project_dir}/models/character_detector_v12_best.pt')

print("✅ Model loaded from Google Drive")
print(f"📁 Path: {project_dir}/models/character_detector_v12_best.pt")

# Validate on test set
metrics = best_model.val()

print("\n📊 YOLOv12 Validation Metrics:")
print(f"   mAP@0.5:      {metrics.box.map50:.3f}  (Target: >0.85)")
print(f"   mAP@0.5:0.95: {metrics.box.map:.3f}   (Target: >0.70)")
print(f"   Precision:    {metrics.box.mp:.3f}     (Target: >0.80)")
print(f"   Recall:       {metrics.box.mr:.3f}     (Target: >0.80)")

if metrics.box.map50 > 0.87:
    print("\n✅ YOLOv12 performance: EXCELLENT (Better than YOLOv8!)")
elif metrics.box.map50 > 0.85:
    print("\n✅ YOLOv12 performance: EXCELLENT")
elif metrics.box.map50 > 0.70:
    print("\n✅ YOLOv12 performance: GOOD")
else:
    print("\n⚠️  Model performance: Acceptable (character detection can improve with fine-tuning)")

## Step 8: Test Inference

In [ ]:
# Test YOLOv12 on sample images
test_results = best_model.predict(
    source='/content/dataset/car plate images/test/images',
    save=True,
    conf=0.5,
    max_det=100,
    project=f'{project_dir}/predictions',
    name='test_predictions'
)

print(f"\n✅ YOLOv12 tested on {len(test_results)} images")
print(f"📁 Results saved to: {project_dir}/predictions/test_predictions")

# Display first result
from IPython.display import Image, display
import glob

result_images = glob.glob(f'{project_dir}/predictions/test_predictions/*.jpg')
if result_images:
    print(f"\nShowing first YOLOv12 prediction:")
    display(Image(filename=result_images[0]))

## Step 9: Export to Core ML (iOS)

In [ ]:
print("📱 Exporting YOLOv12 to Core ML (iOS)...")

# Export YOLOv12 to Core ML
export_path = best_model.export(
    format='coreml',
    imgsz=640,
    nms=True,
    half=False  # FP32 for compatibility
)

print("\n✅ YOLOv12 Core ML export complete!")

# Copy to Google Drive models folder
import shutil
mlpackage_src = export_path  # Path returned by export
mlpackage_dst = f'{project_dir}/models/character_detector_v12.mlpackage'

if os.path.exists(mlpackage_src):
    if os.path.isdir(mlpackage_dst):
        shutil.rmtree(mlpackage_dst)
    shutil.copytree(mlpackage_src, mlpackage_dst)
    print(f"✅ Core ML model saved to: {mlpackage_dst}")
    
    # Check size
    size_mb = sum(os.path.getsize(os.path.join(dirpath, filename))
                  for dirpath, _, filenames in os.walk(mlpackage_dst)
                  for filename in filenames) / (1024 * 1024)
    print(f"📦 Model size: {size_mb:.1f} MB")
else:
    print("⚠️ Export path not found, checking alternatives...")
    !ls -lh {model_dir}/weights/

## Step 10: Export to TensorFlow Lite (Android)

In [ ]:
print("🤖 Exporting YOLOv12 to TensorFlow Lite (Android)...")

# Export YOLOv12 to TFLite FP16
export_path = best_model.export(
    format='tflite',
    imgsz=640,
    int8=False,  # Use FP16
    nms=True
)

print("\n✅ YOLOv12 TFLite export complete!")

# Copy .tflite files to Google Drive
import glob
tflite_files = glob.glob(f'{model_dir}/weights/**/*.tflite', recursive=True)
if tflite_files:
    for tflite_file in tflite_files:
        filename = os.path.basename(tflite_file)
        dst = f'{project_dir}/models/character_detector_v12_{filename}'
        shutil.copy(tflite_file, dst)
        print(f"✅ Copied: {dst}")
        size_mb = os.path.getsize(dst) / (1024 * 1024)
        print(f"   Size: {size_mb:.1f} MB")
else:
    print("ℹ️ Checking export location...")
    !find {model_dir} -name "*.tflite" -type f

## Step 11: Create Download Package

In [ ]:
# Create a final package with all models
import zipfile

# Create zip file in Google Drive
zip_path = f'{project_dir}/trained_models_FINAL.zip'

print("📦 Creating download package...")

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Add all model files
    for root, dirs, files in os.walk(f'{project_dir}/models'):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, f'{project_dir}/models')
            zipf.write(file_path, arcname)
            print(f"   Added: {arcname}")

zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"\n✅ Package created: {zip_path}")
print(f"📦 Size: {zip_size_mb:.1f} MB")
print(f"\n🎉 All your models are in Google Drive!")
print(f"   You can download them anytime from: {project_dir}")

## Step 12: Download Models (Optional)

In [ ]:
# Optional: Download the zip file directly
from google.colab import files

print("⬇️ Starting download...")
print("   (This may take a few minutes depending on file size)")

files.download(zip_path)

print("\n✅ Download complete!")
print("\n📋 What's in the package:")
print("   - character_detector_v12_best.pt (PyTorch)")
print("   - character_detector_v12.mlpackage (iOS Core ML)")
print("   - character_detector_v12_*.tflite (Android TFLite)")
print("\n📁 Models also remain in Google Drive at:")
print(f"   {project_dir}/models/")
print("\n🔒 Your models are safe even if you close Colab!")

## Summary

**✅ Training Complete!**

**What You Have:**
- YOLOv12n model trained on 11,650 images with 51 classes
- Models saved to Google Drive (never lost!)
- Core ML export for iOS
- TFLite export for Android

**Your Models Location:**
```
Google Drive/YOLO_PlateScanner_Training/
├── models/
│   ├── character_detector_v12_best.pt
│   ├── character_detector_v12.mlpackage
│   └── character_detector_v12_*.tflite
├── results/
│   └── uae_character_detector_v12_*/
└── trained_models_FINAL.zip
```

**Next Steps:**
1. Download models from Google Drive (or use the downloaded zip)
2. For iOS:
   - Rename `character_detector_v12.mlpackage` to `character_detector.mlpackage`
   - Add to Xcode project: `PlateScanner/Models/`
3. For Android:
   - Rename `.tflite` file to `character_detector.tflite`
   - Add to Android project: `app/src/main/assets/models/`

**Performance:**
- YOLOv12n is 15-20% faster than YOLOv8
- Better small object detection (characters)
- More compact model size

**🎉 Congratulations! You've trained a state-of-the-art character detection model!**